In [2]:
import re, os, time
from io import BytesIO
from PIL import Image
from pathlib import Path

import requests
from bs4 import BeautifulSoup

import pandas as pd

In [3]:
# def get_session() -> requests.Session:
#     s = requests.Session()
#     s.headers.update(
#         {
#             "User-Agent": "Mozilla/5.0",
#             "Accept-Language": "ko-KR,ko;q=0.9,en;q=0.8",
#             "Referer": RANK_URL,
#             # chunked+압축 조합에서 깨지는 경우가 있어 압축 비활성화(안정성 ↑)
#             "Accept-Encoding": "identity",
#         }
#     )
#     return s

def fetch_html(url: str) -> str:
    res = requests.get(url, timeout=15)
    res.raise_for_status()
    res.encoding = res.apparent_encoding
    return res.text

def get_image_type(content_type: str, first16: bytes) -> str:
    """Content-Type 우선, 모호하면 매직넘버(앞 16바이트)로 판별"""
    ct = (content_type or "").lower().split(";")[0].strip()
    mapping = {
        "image/jpeg": ".jpg",
        "image/jpg": ".jpg",
        "image/png": ".png",
        "image/webp": ".webp",
        "image/gif": ".gif",
        "image/bmp": ".bmp",
        "image/tiff": ".tif",
        "image/svg+xml": ".svg",
    }
    ext = mapping.get(ct, ".img")
    
    if ext != ".img":
        return ext

    # Magic number fallback
    if first16.startswith(b"\xff\xd8\xff"):
        return ".jpg"
    if first16.startswith(b"\x89PNG\r\n\x1a\n"):
        return ".png"
    if first16.startswith(b"GIF87a") or first16.startswith(b"GIF89a"):
        return ".gif"
    if first16.startswith(b"RIFF") and b"WEBP" in first16[:16]:
        return ".webp"
    if first16.startswith(b"BM"):
        return ".bmp"

    return ".img"

def sanitize_filename(name: str, max_len: int = 80) -> str:
    name = re.sub(r"[\\/:*?\"<>|,]", "_", name).strip()
    name = re.sub(r"\s+", "", name)
    return name[:max_len].rstrip(".")

def download_image(img_url, image_name, path="./image") -> str :
    
    while (True) :
        image_response = requests.get(img_url)
        if image_response.status_code == 200 : break
    
    ext = get_image_type(image_response.headers.get("Content-Type", ""), image_response.content[:16])
    img_file = os.path.join(path, sanitize_filename(image_name) + ext)
    
    img = Image.open(BytesIO(image_response.content))
    img.save(img_file)
    
    return img_file

In [4]:
# Main Code
RANK_URL = "https://www.moviechart.co.kr/rank/realtime/index/image"
BASE_URL = "https://www.moviechart.co.kr"
OUT_DIR = Path("./image")

if not os.path.exists(OUT_DIR):
    os.makedirs(OUT_DIR)
  
rank_html = fetch_html(RANK_URL)
soup = BeautifulSoup(rank_html, "html.parser")
    
title_tags = soup.select('ul.movieBox-list div.movie-title a')
image_tags = soup.select('ul.movieBox-list li.movieBox-item > a[href*="/info/movieinfo/detail/"]')

# for image_tag in image_tags :
#     print(f"수집한 이미지 수: {len(title_tags)}")


print(f"수집한 영화 수: {len(title_tags)}, {len(image_tags)}")
    
movies_df = pd.DataFrame(columns=['rank', 'title', 'image'])

for title_tag, image_tag in zip(title_tags, image_tags) :
    
    print('-'*100)
    try :
        title = title_tag.text
        rank = int(image_tag.select_one('p.rank').text)
        
        print("MOVIE :", rank, title)
        
        img_src = image_tag.select_one('img').get('src')
        thumb_url = BASE_URL + img_src
        
        print("thumb_url :", thumb_url) 
        
        img_file = download_image(thumb_url, f"{rank}_"+title, OUT_DIR)
    
        movie = [rank, title, img_file]
        movies_df.loc[rank] = movie
        
        print("MOVIE :", movie)
    
    except Exception as e:
        print(f"[{rank}] {title} ERROR:", e)

print("Done.")

수집한 영화 수: 20, 20
----------------------------------------------------------------------------------------------------
MOVIE : 1 왕과 사는 남자
thumb_url : https://www.moviechart.co.kr/thumb?width=178&height=267&m_code=20242837&source=https://admin.moviechart.co.kr/assets/upload/movie/260108003526_8322.jpg
MOVIE : [1, '왕과 사는 남자', 'image\\1_왕과사는남자.jpg']
----------------------------------------------------------------------------------------------------
MOVIE : 2 휴민트
thumb_url : https://www.moviechart.co.kr/thumb?width=178&height=267&m_code=20241266&source=https://admin.moviechart.co.kr/assets/upload/movie/260112024651_6301.jpg
MOVIE : [2, '휴민트', 'image\\2_휴민트.jpg']
----------------------------------------------------------------------------------------------------
MOVIE : 3 초속 5센티미터
thumb_url : https://www.moviechart.co.kr/thumb?width=178&height=267&m_code=20259583&source=https://admin.moviechart.co.kr/assets/upload/movie/260213052538_7190.jpg
MOVIE : [3, '초속 5센티미터', 'image\\3_초속5센티미터.jpg']
--

In [5]:
movies_df

,rank,title,image
1,1,왕과 사는 남자,image\1_왕과사는남자.jpg
2,2,휴민트,image\2_휴민트.jpg
3,3,초속 5센티미터,image\3_초속5센티미터.jpg
4,4,너자 2,image\4_너자2.jpg
5,5,슬라이드 스트럼 뮤트,image\5_슬라이드스트럼뮤트.jpg
6,6,넘버원,image\6_넘버원.jpg
7,7,햄넷,image\7_햄넷.jpg
8,8,부흥,image\8_부흥.jpg
9,9,매드 댄스 오피스,image\9_매드댄스오피스.jpg
10,10,신의악단,image\10_신의악단.jpg
